# LangChain in 90 Minutes

By the end of this workshop you'll have built, in order:

1. A single call to an LLM
2. A reusable prompt template
3. A chain — steps linked together with `|`
4. Memory — a conversation that remembers what was said
5. **An agent** — a to-do list assistant that decides what to do on its own

Each step is a small twist on the one before it, so nothing here needs a big lecture — including the agent at the end.

## 0. Setup (5 min)

This repo shares one `.env` and one `requirements.txt` at the root (`pip install -r ../requirements.txt`). We just load the key and make sure it works.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

import warnings
warnings.filterwarnings("ignore")  # quiet some noisy internal library warnings for this workshop

# Load the shared .env from the repo root
load_dotenv(Path.cwd().parent / ".env")

model = ChatOpenAI(model="gpt-4o-mini")

# Smoke test — if this prints a reply, your API key is working
response = model.invoke("Say hi in one short sentence.")
print(response.content)

## 1. Talking to the LLM: messages & prompt templates (10 min)

LangChain wraps chat models so you always send a *list of messages*, not just plain text:

- `SystemMessage` — sets the assistant's role or behaviour
- `HumanMessage` — what the user says

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage("You are a friendly assistant who replies in one short sentence."),
    HumanMessage("What's a good icebreaker question for a team meeting?"),
]

response = model.invoke(messages)
print(response.content)

A **prompt template** is a fill-in-the-blanks message list, so you can reuse the same prompt with different inputs instead of retyping it each time.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

email_prompt = ChatPromptTemplate.from_messages([
    ("system", "You write short, friendly work emails."),
    ("human", "Write a 3-sentence email to {recipient} about {topic}."),
])

filled_prompt = email_prompt.invoke({"recipient": "my manager", "topic": "requesting Friday off"})
response = model.invoke(filled_prompt)
print(response.content)

## 2. Chaining steps together with `|` (15 min)

Instead of manually passing a prompt's output into the next step, LangChain lets you connect steps with the `|` operator. `StrOutputParser()` just unwraps the plain text from the model's reply, so you don't need `.content` anymore.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

email_chain = email_prompt | model | StrOutputParser()

result = email_chain.invoke({"recipient": "my manager", "topic": "requesting Friday off"})
print(result)

Now let's link **two** chains: draft a Slack message, then rewrite it in a different tone. The output of the first chain becomes the input to the second — that's chaining.

In [ ]:
draft_prompt = ChatPromptTemplate.from_messages([
    ("system", "You draft short Slack messages."),
    ("human", "Draft a message telling the team {update}."),
])

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", "You rewrite Slack messages to sound more upbeat, keeping them short."),
    ("human", "Rewrite this message: {draft}"),
])

draft_chain = draft_prompt | model | StrOutputParser()
rewrite_chain = rewrite_prompt | model | StrOutputParser()

draft = draft_chain.invoke({"update": "the deploy is delayed by a day"})
print("Draft:", draft)

final_message = rewrite_chain.invoke({"draft": draft})
print("\nRewritten:", final_message)

## 3. Giving it memory: multi-turn conversations (20 min)

An LLM call is stateless — it only ever sees what you send it. To have a *conversation*, you keep a growing list of messages and send the whole list every time.

In [ ]:
from langchain_core.messages import AIMessage

chat_history = [
    SystemMessage("You are a helpful assistant."),
]

def chat(user_text):
    chat_history.append(HumanMessage(user_text))
    response = model.invoke(chat_history)
    chat_history.append(AIMessage(response.content))
    print("You:", user_text)
    print("AI: ", response.content, "\n")

chat("My name is Sonya and I'm planning a trip to Japan.")
chat("What's a good number of days to spend there?")
chat("What's my name, and where am I going again?")

That `chat_history` list *is* memory — nothing fancier is happening under the hood. We just re-send the whole conversation so far on every turn.

Hang on to that idea — we'll reuse the exact same trick for the agent below.

### Optional: try it yourself (skip if short on time)
Uncomment the cell below for a live back-and-forth chat loop (type `exit` to stop).

In [ ]:
# while True:
#     user_input = input("You: ")
#     if user_input.lower() == "exit":
#         break
#     chat(user_input)

## Break (5 min)

Stretch, grab some water — we'll pick back up with agents next.

## 4. From chains to agents (5 min)

So far, **we** decided every step and the order to run them in. An agent flips that: we give the model a set of tools — plain Python functions — and it decides which ones to call, and in what order, based on what's asked.

We're going to build something you'll actually use: a to-do list assistant.

## 5. Build your daily to-do agent (25 min)

A tool is just a normal Python function with a `@tool` decorator and a docstring. The docstring is how the model knows what the tool does and when to use it.

In [ ]:
from langchain_core.tools import tool

tasks: list[str] = []

@tool
def add_task(task: str) -> str:
    """Add a new task to the to-do list."""
    tasks.append(task)
    return f"Added: {task}"

@tool
def list_tasks() -> str:
    """List all current tasks, numbered."""
    if not tasks:
        return "No tasks yet."
    return "\n".join(f"{i + 1}. {t}" for i, t in enumerate(tasks))

@tool
def complete_task(task_number: int) -> str:
    """Mark a task as complete and remove it, using its number from list_tasks."""
    if 1 <= task_number <= len(tasks):
        done = tasks.pop(task_number - 1)
        return f"Completed: {done}"
    return f"No task numbered {task_number}."

Now hand those tools to `create_react_agent` along with a model. That's the whole agent.

In [ ]:
from langgraph.prebuilt import create_react_agent

todo_agent = create_react_agent(model, tools=[add_task, list_tasks, complete_task])

We drive it the exact same way we drove memory in section 3: keep a growing `messages` list and hand the whole thing to the agent each turn. Run these one at a time and watch it remember tasks it added earlier.

In [ ]:
agent_messages = [HumanMessage("Add 'buy milk' and 'call mom' to my list.")]
result = todo_agent.invoke({"messages": agent_messages})
agent_messages = result["messages"]
print(agent_messages[-1].content)

In [ ]:
agent_messages.append(HumanMessage("What's on my list?"))
result = todo_agent.invoke({"messages": agent_messages})
agent_messages = result["messages"]
print(agent_messages[-1].content)

In [ ]:
agent_messages.append(HumanMessage("Mark the first one done."))
result = todo_agent.invoke({"messages": agent_messages})
agent_messages = result["messages"]
print(agent_messages[-1].content)

In [ ]:
agent_messages.append(HumanMessage("What's on my list now?"))
result = todo_agent.invoke({"messages": agent_messages})
agent_messages = result["messages"]
print(agent_messages[-1].content)

## 6. Recap & where to go next (5 min)

**The arc we just built:**

1. A single call to an LLM
2. A reusable prompt template
3. A chain — steps linked with `|`
4. Memory — a growing list of messages
5. **An agent** — same message list, except the model also picks which tool to call

**Deliberately not covered today** (see this repo for more):
- Retrieval-Augmented Generation (RAG) → `langchain_fundamentals/rag_systems.py`
- Structured output & multi-agent workflows → `vibecoder/`

**To keep building after today:**
```bash
pip install -r ../requirements.txt   # shared across this whole repo
cp ../.env.example ../.env           # then add your OPENAI_API_KEY
```